# 聚类算法（Clustering）系统学习笔记

---

## 目录

1. [聚类概述与无监督学习](#1-聚类概述与无监督学习)
2. [K-Means 算法](#2-k-means-算法)
3. [DBSCAN 算法](#3-dbscan-算法)
4. [K-Means vs DBSCAN 对比](#4-k-means-vs-dbscan-对比)
5. [聚类算法如何评估与调参](#5-聚类算法如何评估与调参)
6. [总结与速查](#6-总结与速查)

---

## 1. 聚类概述与无监督学习

### 1.1 什么是聚类

> **聚类：把相似的东西分到一组。**

聚类是一种**无监督学习**（Unsupervised Learning）问题，与前面学过的回归、分类有本质区别：

| | 监督学习（回归/分类） | 无监督学习（聚类） |
|:---|:---|:---|
| **是否有标签** | 有标签 $y$ | **没有标签** |
| **目标** | 学习输入到输出的映射 | 发现数据内在结构 |
| **典型任务** | 预测额度、判断类别 | 客户分群、文档归类 |
| **评估难度** | 容易（有标准答案） | 困难（无标准答案） |

### 1.2 无监督问题的核心难点

> 课件强调：「无监督问题：我们手里没有标签了。**难点：如何评估，如何调参。**」

- **如何评估**：没有真实标签，无法用准确率等指标直接衡量好坏
- **如何调参**：参数（如 K、ϵ）的选择没有"对错"，只能靠经验、业务理解和内部指标

### 1.3 聚类的基本思想

聚类希望达成：**簇内相似度高（紧凑），簇间相似度低（分离）。**

```
   ● ● ●         ○ ○
   ● ● ●         ○ ○ ○
   簇1（紧凑）      簇2（分离良好）
```

---

## 2. K-Means 算法

### 2.1 基本概念

K-Means 是最经典的**基于划分**的聚类算法，核心是"均值"（means）。

**两个关键输入：**

1. **K 值**：要得到簇的个数，需要**人为指定 K 值**
2. **距离度量**：如何衡量"相似"

### 2.2 距离度量（补充基础）

| 度量方式 | 公式 | 特点 | 适用场景 |
|:---|:---|:---|:---|
| **欧几里得距离** | $d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$ | 直线距离，最常用 | 数值型、量纲统一的特征 |
| **余弦相似度** | $\cos\theta = \frac{x \cdot y}{\|x\| \cdot \|y\|}$ | 只看方向，不看大小 | 文本、高维稀疏数据 |

> 计算相似度前要**先标准化**。


### 2.3 质心（Centroid）

> 质心 = 均值，即向量各维取平均即可。

对于簇 $C_k$，其质心为：

$$\mu_k = \frac{1}{|C_k|} \sum_{x \in C_k} x$$

例如一个二维簇内有三个点 (1,2)、(3,4)、(5,6)，质心为 $((1+3+5)/3,\ (2+4+6)/3) = (3, 4)$。

### 2.4 优化目标

K-Means 的优化目标是最小化**簇内平方和**（Sum of Squared Errors, SSE）：

$$J = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

> 直观理解：让每个点到它所属簇的质心的距离平方和尽可能小，即簇内尽可能紧凑。

### 2.5 工作流程（核心算法步骤）

1. 随机初始化K个质心：$μ₁, μ₂, ..., μ_K$
2. 重复以下两步直到收敛：
   a. 分配（Assignment）：把每个样本分配到距离最近的质心所属的簇
      $$C_k = { x : ||x - μ_k||² ≤ ||x - μ_j||², 对任意 j ≠ k }$$
   b. 更新（Update）：重新计算每个簇的质心
      $$μ_k = (1/|C_k|) Σ_{x ∈ C_k} x$$
3. 当质心不再变化（或变化极小）时停止

**为什么一定能收敛？** 因为每次分配和更新都会让目标函数 $J$ 单调下降，而 $J$ 有下界（≥0），所以算法必然收敛。但可能收敛到**局部最优**而非全局最优。

### 2.6 K 值如何确定（补充基础）

在算法中，通常认为"K值难确定"，几种常见方法有：

1. **肘部法则（Elbow Method）**：画出 K 与 SSE 的关系曲线，找到"拐点"（肘部）作为 K
2. **轮廓系数（Silhouette Coefficient）**：取值在 [-1, 1]，越接近 1 聚类质量越好，取最大时的 K
3. **业务经验**：根据实际业务需求（如"把客户分成高/中/低三档"）直接定 K

### 2.7 优劣势

| 优势 | 劣势 |
|:---|:---|
| 简单、快速，适合常规数据集 | **K 值难确定** |
| 复杂度与样本呈**线性关系** $O(n \cdot K \cdot t)$ | **对初始质心敏感（可能陷入局部最优）** |
| 容易理解和实现 | **很难发现任意形状的簇**（只能发现凸的、球形的簇） |
| 可扩展性好 | 对噪声和离群点敏感 |


---

## 3. DBSCAN 算法

### 3.1 基本概念

DBSCAN（**D**ensity-**B**ased **S**patial **C**lustering of **A**pplications with **N**oise）是一种**基于密度**的聚类算法。

> 核心思想：**把高密度区域连成一片形成簇，低密度区域的点视为噪声。**

### 3.2 两个关键参数

| 参数 | 含义 |
|:---|:---|
| **ϵ（epsilon）** | 邻域的距离阈值（半径 r） |
| **MinPts** | 密度阈值（一个点成为核心点所需的最小邻居数） |

### 3.3 点的三种类型（核心概念）

| 类型 | 定义 | 通俗理解 |
|:---|:---|:---|
| **核心点（Core Point）** | 该点在 ϵ 邻域内的点的数量 ≥ MinPts | 密度达到阈值，"发展下线"的中心 |
| **边界点（Border Point）** | 在某个核心点的 ϵ 邻域内，但自身邻域点数 < MinPts | 属于某个簇，但**不能发展下线**了 |
| **噪声点（Noise Point）** | 不属于任何簇，从任何核心点都密度不可达 | 离群点，被算法丢弃 |

### 3.4 三个核心关系（务必理解）

设 ϵ 为邻域半径，MinPts 为密度阈值：

1. **直接密度可达（Directly Density-Reachable）**
   - 点 $p$ 在点 $q$ 的 ϵ 邻域内，**且 $q$ 是核心点**，则 $p$ 从 $q$ 直接密度可达
   - 注意：这是**不对称**的——核心点可以"到达"邻居，但边界点（非核心）不能反向到达

2. **密度可达（Density-Reachable）**
   - 存在一个点序列 $q_0, q_1, \dots, q_k$，对任意相邻的 $q_{i-1} \to q_i$ 都是直接密度可达
   - 本质是直接密度可达的"**传播**"（链条式传递）

3. **密度相连（Density-Connected）**
   - 若从某个核心点 $p$ 出发，点 $q$ 和点 $k$ 都是密度可达的，则 $q$ 和 $k$ 密度相连
   - 用于把多个密度可达的点**合并成同一个簇**

> 关系总结：直接密度可达 →（传播）→ 密度可达 →（合并）→ 密度相连 →（成簇）

### 3.5 工作流程

```
参数：数据集 D，半径 ϵ，密度阈值 MinPts
1. 标记所有点为"未访问"
2. 对每个未访问点 p：
   a. 标记 p 为已访问
   b. 找出 p 的 ϵ 邻域内所有点
   c. 若邻域点数 ≥ MinPts：
      - p 是核心点，创建一个新簇 C，把 p 加入 C
      - 扩展：把 p 邻域内密度可达的所有点加入 C
   d. 否则：暂时标记 p 为噪声点
3. 重复直到所有点都被访问
4. 最终未被归入任何簇的点 = 噪声点（离群点）
```

### 3.6 参数选择（补充基础）

> 课件强调：「参数难以选择，参数对结果的影响非常大。」

**半径 ϵ 的选择 —— 基于 K-距离：**

- **K-距离**：给定数据集 $P$，计算点 $p(i)$ 到所有其他点的距离并从小到大排序，第 $k$ 近的距离 $d(k)$ 就是该点的 k-距离
- **方法**：把所有点的 k-距离画成**排序曲线**，寻找**突变点（拐点）**作为 ϵ
- **MinPts**：k-距离中 k 的值，一般取小一些，**多次尝试**调整

> 经验法则：MinPts 通常 ≥ 数据维度数，常用 4~10；ϵ 配合 K-距离图选取拐点。

### 3.7 优劣势

| 优势 | 劣势 |
|:---|:---|
| **不需要指定簇个数**（自动发现簇） | **高维数据较困难**（可先降维） |
| **擅长找到离群点**（适合异常检测任务） | Sklearn 中**效率较慢**（可做数据削减策略） |
| **可以发现任意形状的簇**（非球形） | 参数难以选择（对结果影响非常大） |
| 只需两个参数 | 密度差异大时效果差 |

---

## 4. K-Means vs DBSCAN 对比

| 维度 | K-Means | DBSCAN |
|:---|:---|:---|
| **聚类依据** | 基于距离（划分） | 基于密度 |
| **是否需要指定簇数** | 需要指定 K | 不需要（自动发现） |
| **簇的形状** | 只能发现**凸的/球形**簇 | 能发现**任意形状**簇 |
| **对噪声/离群点** | 敏感 | 不敏感（能识别为噪声点） |
| **参数** | 1 个（K） | 2 个（ϵ, MinPts） |
| **高维数据** | 表现尚可 | 较困难 |
| **效率** | 快，线性复杂度 | 较慢 |
| **适用场景** | 数据大致球形分布、已知簇数 | 数据形状任意、需检测离群点 |

---

## 5. 聚类算法如何评估与调参

由于无监督问题没有标签，评估与调参是最大难点。常用方法：

### 5.1 内部指标（无需标签）

| 指标 | 含义 | 取值说明 |
|:---|:---|:---|
| **轮廓系数（Silhouette）** | 结合簇内紧凑度与簇间分离度 | [-1, 1]，越接近 1 越好 |
| **DB 指数（Davies-Bouldin）** | 簇间相似度与簇内相似度的比值 | 越小越好 |
| **SSE（簇内平方和）** | 簇内点到质心距离平方和 | 越小越好（用于肘部法则） |

### 5.2 调参策略

- **K-Means 调 K**：画 SSE-K 曲线找肘部，或画轮廓系数-K 曲线取峰值
- **DBSCAN 调 ϵ 和 MinPts**：用 K-距离图找 ϵ 的拐点，MinPts 多次尝试
- **可视化验证**：直观查看聚类效果（尤其二维数据）

### 5.3 推荐的可视化工具

> 课件提供两个极好的交互式可视化链接：
> - DBSCAN：https://www.naftaliharris.com/blog/visualizing-dbscan-clustering/
> - K-Means：https://www.naftaliharris.com/blog/visualizing-k-means-clustering/

强烈建议动手玩一遍，直观理解算法行为。

---

## 6. 总结与速查

### 6.1 一句话总结

| 算法 | 核心思想 | 一句话 |
|:---|:---|:---|
| **K-Means** | 均值迭代 | 指定 K 个质心，反复"分配-更新"直到收敛，最小化簇内平方和 |
| **DBSCAN** | 密度可达 | 用 ϵ 和 MinPts 定义核心点，通过密度可达传播连接成簇，其余为噪声 |

### 6.2 关键公式速查

| 名称 | 公式 |
|:---|:---|
| 欧氏距离 | $d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$ |
| 余弦相似度 | $\cos\theta = \frac{x \cdot y}{\|x\| \cdot \|y\|}$ |
| K-Means 质心 | $\mu_k = \frac{1}{\|C_k\|}\sum_{x \in C_k} x$ |
| K-Means 目标函数（SSE） | $J = \sum_{k=1}^{K}\sum_{x \in C_k}\|x - \mu_k\|^2$ |
| DBSCAN 核心点条件 | $\|N_\epsilon(p)\| \ge MinPts$ |

### 6.3 算法选择建议

```
数据大致球形分布、已知簇数？ → K-Means
需要检测离群点 / 数据形状任意？ → DBSCAN
高维数据？ → 先降维，再选算法
不确定？ → 两者都试，用轮廓系数等内部指标比较
```

---

*笔记基于课件《4-聚类算法》整理与扩展，补充了距离度量、优化目标、算法流程、参数选择与评估等基础内容。*
